In [1]:
import os
import pandas as pd
import polars as pl
import numpy as np
import torch 
import sys
from pathlib import Path
import copy
import torch.nn.functional as F
import torch
import torch.nn as nn
import kaggle_evaluation.nfl_inference_server
sys.path.append("/kaggle/input/nfl2026-dataset/code")


## Constant values and pre-computed feature statistics (same with train)

In [2]:
# constant values
FIELD_WIDTH: float = 120.0
FIELD_HEIGHT: float = 53.3
MAX_PLAYER_NUM: int = 22
MAX_IN_PLAYER_NUM: int = 22
MAX_OUT_PLAYER_NUM: int = 22
player_roles = ['Defensive Coverage', 'Targeted Receiver', 'Passer', 'Other Route Runner']
player_role_to_idx = {role: idx for idx, role in enumerate(player_roles)}
src_feat_names = ["x", "y", "s", "a", "dir", "o"]
src_feat_idx = {name: i for i, name in enumerate(src_feat_names)}

# Pre-computed feature statistics
feature_mean = {'x': 60.82776974002471,
                'y': 26.66586819309761,
                'ori_sin': -0.028771674664162084,
                'ori_cos': -0.0860770011312027,
                'distance_to_ball': 19.420012294946886,
                'distance_to_receiver': 13.905649795712955,
                'ball_land_x_dx': 4.807716986443858,
                'ball_land_y_dy': -0.6598223350763681,
                'receiver_dx': -3.034959448458881,
                'receiver_dy': -0.3004955149788581,
                'dir_sin_s': -0.04365514987349996,
                'dir_cos_s': 1.7474138934015886,
                'dir_sin_a': -0.022489013627792452,
                'dir_cos_a': 1.0584119108147563,
                'diff_dir_o_sin': -0.0064997680282648995,
                'diff_dir_o_cos': 0.1363704308055112,
                'player_role_encoded': 1.1193698534538628,
                'num_frame_output': 11.64147,
                'passer_x': 49.05013399503722,
                'passer_y': 26.528289259127966,
                'ball_land_x': 65.33169440307206,
                'ball_land_y': 26.221190352646918,
                'rot_x': -12.97965036,
                'rot_y': 0.54228895,
                'dir_sin': -0.008201030098231301,
                'dir_cos': 0.45047703362132063,
                'a': 2.118334968043749,
                's': 3.0198784693373515,
                'diff_x': 0.1789712431138102,
                'diff_y': -0.004446990066127387,
                'player_last_x': 60.82776974002471,
                'player_last_y': 26.66586819309761,
                'passer_last_x': 49.05013399503722,
                'passer_last_y': 26.528289259127966,
                }


feature_std = {'x': 23.479941313323085,
               'y': 10.00749618625406,
               'ori_sin': 0.543936436139763,
               'ori_cos': 0.8342039901959803,
               'distance_to_ball': 10.76441761817005,
               'distance_to_receiver': 8.820358107267353,
               'ball_land_x_dx': 12.124398080266454,
               'ball_land_y_dy': 17.957146201379818,
               'receiver_dx': 7.586012961533925,
               'receiver_dy': 14.293949140615915,
               'dir_sin_s': 2.29624100410567,
               'dir_cos_s': 2.399019308047086,
               'dir_sin_a': 1.5753721593310632, 'dir_cos_a': 1.699787386078158,
               'diff_dir_o_sin': 0.6532995066440124,
               'diff_dir_o_cos': 0.7446884008166831,
               'player_role_encoded': 1.3341219458609044,
               'num_frame_output': 5.3315372,
               'passer_x': 23.15646164494253,
               'passer_y': 4.039143779870031,
               'ball_land_x': 24.721183980071213,
               'ball_land_y': 15.253264763883333,
               'rot_x': 12.98279418,
               'rot_y': 12.47606231,
               'dir_sin': 0.6209186583613782,
               'dir_cos': 0.6414541045817329,
               'a': 1.4157939116699676,
               's': 2.227938568313042,
               'diff_x': 0.24064030768643296,
               'diff_y': 0.23067443165381496,
               'player_last_x': 23.479941313323085,
               'player_last_y': 10.00749618625406,
               'passer_last_x': 23.15646164494253,
               'passer_last_y': 4.039143779870031,
               }


target_mean = {'x': 69.50731170150776,
               'y': 26.49108062017707,
               'dx': 2.4863890033680556,
               'dy': -0.0003338034874301969,
               'x0': 67.02092269813976,
               'y0': 26.49141442366451,
               'ball_land_x_dx': 2.1796843109462185,
               'ball_land_y_dy': 0.01456915004472861,
               'sx': 0.24357714,
               'sy': -0.00259809,
               'ax': -0.00855656,
               'ay': 0.00024983,
               }
target_std = {'x': 23.435529312124537,
              'y': 13.445279179127226,
              'dx': 3.9917733048457364,
              'dy': 3.728388769750037,
              'x0': 22.991527135040435,
              'y0': 11.787700462682832,
              'ball_land_x_dx': 7.819356803570175,
              'ball_land_y_dy': 7.527393631406031,
              'sx': 0.30963882,
              'sy': 0.42498858,
              'ax': 0.02965129,
              'ay': 0.0291162,

              }
player_xy_mean = (60.82776974002471, 26.66586819309761)
player_xy_std = (23.479941313323085, 10.00749618625406)
passer_xy_mean = (49.05013399503722, 26.528289259127966)
passer_xy_std = (23.15646164494253, 4.039143779870031)
ball_land_xy_mean = (65.33169440307206, 26.221190352646918)
ball_land_xy_std = (24.721183980071213, 15.253264763883333)


print(MAX_IN_PLAYER_NUM)
print(MAX_OUT_PLAYER_NUM)

22
22


## Configuration (same with train)

In [3]:

class DictDotNotation(dict):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.__dict__ = self


CFG = DictDotNotation(
    batchsize1=64,
    batchsize2=64,
    num_workers1=2,
    num_workers2=2,
    epoch=35,
    train_data_repeat_factor=6,
    start_valid_epoch=8,
    n_fold=5,
    seed=0,
    device="cuda:0",

    # amp1=True,
    # amp1_dtype="float16",
    # amp2=False,

    input_sequence_size=20,
    output_sequence_size=48,

    model_param=DictDotNotation(
        dim1=256,
        dim2=32,
        dropout0=0.0,
        dropout1=0.0,
        dropout2=0.0,
        input_dim_multiplier=64,
        # register_num=0,
        # register_sigma=1.0,
        tfm_nhead=8,
        tfm_dropout=0.05,
        dim_feedforward=1024,
        num_layers=3,
        min_var=0.001,
        aux_loss_weight=0.8,
        aux_target_dim=4,
        static_feature_dim=12,
    ),


    # optimizer  params
    lr=0.0005,
    weight_decay=1e-4,
    grad_max_norm=200.0,
    grad_max_value=None,
    ema_decay=0.9995,
    ema_update_after_epoch=5,

    # data augumentation params
    angle_std_rate=0.0,
    angle_uni_rate=0.5,
    angle_std=1.0,
    x_bias_std=1.0,
    y_bias_std=1.5,

    flipud_rate=0.5,
    max_shift_size=20,
    frame_shift_rate=0.675,

    frame_scale_rate=0.0,
    frame_scale_std=0.0,

    persistent_workers=False,
    pin_memory=True,
    save_model=True,

    target_names=["dx",
                  "dy",
                  ],
    fold_seeds = (0,1,2,3,4,5,6,7,8,9),  # If computational resources allow, it is better to evaluate with (0, 1, 2)
)


In [4]:

field_mergine = 0.0 # Threshold for clipping values outside the field
input_size = 20

target_mean_xy = [target_mean[key] for key in ("dx", "dy")]
target_std_xy = [target_std[key] for key in ("dx", "dy")]

target_mean_xy = np.array(target_mean_xy).reshape(2, 1, 1)
target_std_xy = np.array(target_std_xy).reshape(2, 1, 1)


## Model

In [5]:
class Block(nn.Module):
    def __init__(self, dim, kernel_size=3, groups=1, padding=0, actf=nn.SiLU()):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(dim),
            actf,
            nn.Conv1d(dim, dim, kernel_size=kernel_size, groups=groups, padding=padding, bias=False),
            nn.BatchNorm1d(dim),
            actf,
            nn.Conv1d(dim, dim, kernel_size=kernel_size, groups=groups, padding=padding, bias=False),
        )

    def forward(self, x):
        return self.block(x)


class ResBlock(nn.Module):
    def __init__(self, dim, kernel_size=3, groups=1, actf=nn.SiLU()):
        padding = (kernel_size - 1) // 2
        super().__init__()
        self.block = Block(
            dim=dim,
            kernel_size=kernel_size,
            padding=padding,
            groups=groups,
            actf=actf
        )

    def forward(self, x):
        return x + self.block(x)


class SequenceStem(nn.Module):
    def __init__(self, input_dim, nbai, kernel_size=3):
        super().__init__()
        dim = input_dim * nbai

        # Depthwise convolution
        self.stem = nn.Conv1d(
            input_dim, dim,
            kernel_size=kernel_size,
            groups=input_dim,
            bias=False
        )

        # Three residual blocks (with different group settings)
        self.blocks = nn.ModuleList([
            Block(dim, kernel_size, padding=0, groups=input_dim),
            Block(dim, kernel_size, padding=0, groups=input_dim),
            Block(dim, kernel_size, padding=0, groups=input_dim // 2),
        ])

        # No padding is used in the convolution,
        # so with kernel_size=3 the output length is reduced by 2
        # (this is related to x[:, :, 4:] in the forward pass)
        self.crop = (kernel_size - 1) * 2  # e.g., 3 → 4

    def forward(self, x):
        x = self.stem(x)

        # Residual blocks (add after cropping)
        for block in self.blocks:
            x = x[:, :, self.crop:] + block(x)

        return x


class MyTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead,
                 dim_feedforward=2048,
                 dropout=0.1,
                 batch_first=True,
                 norm_first=True):

        super().__init__()

        self.batch_first = batch_first
        self.norm_first = norm_first

        # --- Multi-Head Attention ---
        self.self_attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=batch_first,
        )

        self.linear1_w1 = nn.Linear(d_model, dim_feedforward)
        self.linear1_w2 = nn.Linear(d_model, dim_feedforward)

        self.linear2 = nn.Linear(dim_feedforward, d_model)

        # --- LayerNorm ---
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # --- Dropout ---
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def _ffn_swiglu(self, x):
        """
        SwiGLU(x) = (W1 x) * silu(W2 x)
        """
        a = self.linear1_w1(x)
        b = F.silu(self.linear1_w2(x))
        x = a * b
        return self.linear2(self.dropout(x))

    def forward(self, src, src_mask=None, src_key_padding_mask=None):

        # ============================
        #      PRE-NORM
        # ============================
        if self.norm_first:
            # --- LayerNorm → MHA → res ---
            src2 = self.norm1(src)
            attn_out, _ = self.self_attn(
                src2, src2, src2,
                attn_mask=src_mask,
                key_padding_mask=src_key_padding_mask,
                need_weights=False,
            )
            src = src + self.dropout1(attn_out)

            # --- LayerNorm → SwiGLU FFN → res ---
            src2 = self.norm2(src)
            src2 = self._ffn_swiglu(src2)
            src = src + self.dropout2(src2)

        # ============================
        #      POST-NORM
        # ============================
        else:
            # --- MHA → res → LayerNorm ---
            attn_out, _ = self.self_attn(
                src, src, src,
                attn_mask=src_mask,
                key_padding_mask=src_key_padding_mask,
                need_weights=False,
            )
            src = self.norm1(src + self.dropout1(attn_out))

            # --- SwiGLU FFN → res → LayerNorm ---
            src2 = self._ffn_swiglu(src)
            src = self.norm2(src + self.dropout2(src2))

        return src




class MyTransformerEncoder(nn.Module):
    def __init__(self, layer, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(num_layers)])

    def forward(self, x, src_mask=None, src_key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, src_mask=src_mask, src_key_padding_mask=src_key_padding_mask)
        return x


class Head(nn.Module):
    def __init__(self, dim2, aux_target_dim, kernel_size=3):
        super().__init__()
        self.dim2 = dim2
        self.aux_target_dim = aux_target_dim

        self.model = nn.Sequential(
            ResBlock(self.dim2, kernel_size=kernel_size),
            ResBlock(self.dim2, kernel_size=kernel_size),
        )
        self.head_mu = nn.Sequential(
            nn.Conv1d(self.dim2, 2, kernel_size=1)
        )
        self.head_var = nn.Sequential(
            nn.Conv1d(self.dim2, 2, kernel_size=1)
        )
        self.head_aux_mu = nn.Sequential(
            nn.Conv1d(self.dim2, self.aux_target_dim, kernel_size=1)
        )
        self.head_aux_var = nn.Sequential(
            nn.Conv1d(self.dim2, self.aux_target_dim, kernel_size=1)
        )

    def __reshape_permute(self, x, bsize, num_player):
        # b*p,c,l -> b,c,p,l
        _, C, L = x.shape
        x = torch.permute(torch.reshape(x, (bsize, num_player, C, L)), (0, 2, 1, 3))  # b 2 p l
        return x

    def forward(self, z, bsize, num_player):

        z = self.model(z)  # bp c L
        pred_m = self.head_mu(z)  # bp 2 L
        pred_v = self.head_var(z)  # bp 2 L
        auxi_m = self.head_aux_mu(z)
        auxi_v = self.head_aux_var(z)

        pred_m = self.__reshape_permute(pred_m, bsize, num_player)
        pred_v = self.__reshape_permute(pred_v, bsize, num_player)
        auxi_m = self.__reshape_permute(auxi_m, bsize, num_player)
        auxi_v = self.__reshape_permute(auxi_v, bsize, num_player)

        return pred_m, pred_v, auxi_m, auxi_v


class Model(nn.Module):
    def __init__(self, input_dim, output_sequence_size, loss_fn, model_param):
        super().__init__()
        self.dim1 = model_param.dim1
        self.dim2 = model_param.dim2
        self.input_dim_multiplier = model_param.input_dim_multiplier
        self.tfm_nhead = model_param.tfm_nhead
        self.dim_feedforward = model_param.dim_feedforward
        self.tfm_dropout = model_param.tfm_dropout
        self.num_layers = model_param.num_layers
        self.dropout0 = model_param.dropout0
        self.dropout1 = model_param.dropout1
        self.dropout2 = model_param.dropout2
        self.aux_loss_weight = model_param.aux_loss_weight
        self.aux_target_dim = model_param.aux_target_dim
        self.static_feature_dim = model_param.static_feature_dim


        dim0 = input_dim * self.input_dim_multiplier + self.input_dim_multiplier

        self.sequence_stem = SequenceStem(input_dim, self.input_dim_multiplier)
        self.feature_stem = nn.Sequential(nn.Conv1d(self.static_feature_dim, self.input_dim_multiplier, kernel_size=1, bias=False),
                                          nn.BatchNorm1d(self.input_dim_multiplier),
                                          nn.SiLU(),
                                          )


        self.merge_layer = nn.Sequential(
            nn.Conv1d(dim0, self.dim1, kernel_size=1, bias=False),
            nn.BatchNorm1d(self.dim1),
            ResBlock(self.dim1, kernel_size=1),
            ResBlock(self.dim1, kernel_size=1),
            nn.BatchNorm1d(self.dim1),
        )

        encoder_layer = MyTransformerEncoderLayer(d_model=self.dim1, nhead=self.tfm_nhead,
                                                  dim_feedforward=self.dim_feedforward,
                                                  dropout=self.tfm_dropout, batch_first=True)
        self.encoder = MyTransformerEncoder(encoder_layer, num_layers=self.num_layers)



        self.pred_sequence_layer = nn.Sequential(
            nn.Conv1d(self.dim1, output_sequence_size * self.dim2, kernel_size=1)
        )

        self.head = Head(self.dim2, self.aux_target_dim)
        self.output_sequence_size = output_sequence_size
        self.loss_fn = loss_fn
        self.min_var = model_param.min_var


    def __calc_loss__(self, pred_m, pred_v, auxi_m, auxi_v, target_xy, target_sa, target_mask):

        pred_loss = self.loss_fn(pred_m, target_xy, pred_v)
        pred_loss = (pred_loss * target_mask).sum() / target_mask.sum()


        aux_loss = self.loss_fn(auxi_m, target_sa, auxi_v)
        aux_loss = aux_loss * (2.0 / aux_loss.shape[1])  # Scale to keep the balance with the main loss
                                                         # even if the number of auxiliary losses increases
        aux_loss = (aux_loss * target_mask[:, :1]).sum() / target_mask.sum()

        loss = (1.0 - self.aux_loss_weight) * pred_loss + self.aux_loss_weight * aux_loss

        return loss

    def __proc(self, data):
        # self = model
                # self = model
        x_dynamic = data["dynamic_feat"]  # b c p l
        x_static = data["static_feat"]  # b p c

        bsize, c, num_player, l = x_dynamic.shape
        x_dynamic = torch.permute(x_dynamic, (0, 2, 1, 3))  # b p c l
        x_dynamic = torch.reshape(x_dynamic, (bsize * num_player, c, l))  # bp c l

        x_static = torch.reshape(x_static, (bsize * num_player, -1, 1))  # b p c -> bp c 1

        x_dynamic = self.sequence_stem(x_dynamic)
        x_static = self.feature_stem(x_static)

        # crop last frame
        x_dynamic = x_dynamic[:, :, -1:]  # bp c 1
        x = torch.concat((x_dynamic, x_static), dim=1)
        x = self.merge_layer(x)  # bp c 1
        x = x.reshape(bsize, num_player, self.dim1)  # b num_player c

        key_padding_mask = torch.arange(num_player, device=x.device).expand(bsize, num_player) >= data["num_player_input"].unsqueeze(1).to(device=x.device)  # shape (B, L)
        x = x.contiguous()
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        x = x.reshape(bsize * num_player, self.dim1, 1)  # b p c

        x = self.pred_sequence_layer(x)  # bp C 1
        z = x.reshape(-1, self.dim2, self.output_sequence_size)  # bp c L

        pred_m, pred_v, auxi_m, auxi_var = self.head(z, bsize, num_player)
        return pred_m, pred_v, auxi_m, auxi_var


    def forward(self, data, calc_loss=True, return_pred=True):
        # self = model

        pred_m, pred_v, auxi_m, auxi_var = self.__proc(data)
        
        output_dict = {}
        if calc_loss:
            target_xy = data["target_xy"]  # b c p l
            target_sa = data["target_sa"]  # b c p l
            target_mask = data["target_mask"]  # b c p l

            pred_v = torch.nn.Softplus()(pred_v) + self.min_var
            auxi_var = torch.nn.Softplus()(auxi_var) + self.min_var
            loss = self.__calc_loss__(pred_m, pred_v, auxi_m, auxi_var, target_xy, target_sa, target_mask)
            output_dict["loss"] = loss

        # Collect per-player predictions with the original output frame length
        preds = []
        if return_pred:
            bsize = len(pred_m)
            for i in range(bsize):
                # Retrieve the number of output frames and players for this sample
                num_frame_output = data["num_frame_output"][i]
                num_player_output = data["num_player_output"][i]
                for j in range(num_player_output):
                    # Extract valid frames only, detach from the graph, and move to CPU
                    pred = pred_m[i, :, j, :num_frame_output].detach().cpu().numpy()
                    preds.append(pred)
        # Store collected predictions in the output dictionary
        output_dict["preds"] = preds
        return output_dict

    def inference(self, data):
        pred_m, pred_v, auxi_m, auxi_var = self.__proc(data)
        return pred_m



## Dataset for test

In [6]:
class TestDataset(torch.utils.data.Dataset):
    def __init__(self,
                 test_input,
                 test,
                 CFG,
                 ):
        self.test_input = test_input.copy()
        self.test = test.copy()

        self.input_sequence_size = CFG.input_sequence_size
        self.output_sequence_size = CFG.output_sequence_size
        self.ext_feat_names = CFG.ext_feat_names.copy()
        self.ext_feat_idx = {name: i for i, name in enumerate(self.ext_feat_names)}

        # augumentation parametors
        self.x_bias_std = CFG.x_bias_std
        self.y_bias_std = CFG.y_bias_std
        self.flipud_rate = CFG.flipud_rate
        self.angle_uni_rate = CFG.angle_uni_rate
        self.angle_std_rate = CFG.angle_std_rate
        self.angle_std = CFG.angle_std

        self.max_shift_size = CFG.max_shift_size
        self.least_input_frame = 5
        self.frame_shift_rate = CFG.frame_shift_rate

        # feature/target statistics
        self.feat_mean = np.array([feature_mean[name] for name in self.ext_feat_names])
        self.feat_std = np.array([feature_std[name] for name in self.ext_feat_names])
        self.target_mean = np.array([target_mean[name] for name in ("dx", "dy")])
        self.target_std = np.array([target_std[name] for name in ("dx", "dy")])

        self.target_sa_mean = np.array([target_mean[name] for name in ("sx", "sy", "ax", "ay")])
        self.target_sa_std = np.array([target_std[name] for name in ("sx", "sy", "ax", "ay")])

        self.player_last_xy_mean = np.array([feature_mean[name] for name in ("player_last_x", "player_last_y")])
        self.player_last_xy_std = np.array([feature_std[name] for name in ("player_last_x", "player_last_y")])

        self.passer_last_xy_mean = np.array([feature_mean[name] for name in ("passer_last_x", "passer_last_y")])
        self.passer_last_xy_std = np.array([feature_std[name] for name in ("passer_last_x", "passer_last_y")])

        self.ball_land_xy_mean = np.array([feature_mean[name] for name in ("ball_land_x", "ball_land_y")])
        self.ball_land_xy_std = np.array([feature_std[name] for name in ("ball_land_x", "ball_land_y")])

        self.num_frame_output_mean = np.array([feature_mean[name] for name in ("num_frame_output",)])
        self.num_frame_output_std = np.array([feature_std[name] for name in ("num_frame_output",)])


    def __len__(self):
        return 1

    def target_scaling(self, target):
        return (target - self.target_mean) / self.target_std

    def feature_scaling(self, feature):
        return (feature - self.feat_mean) / self.feat_std

    def target_rescaling(self, target):
        return (target * self.target_std) + self.target_mean


    def feat_pad(self, x, max_num_player=MAX_PLAYER_NUM, input_sequence_size=20, val=0.0):

        assert x.shape[0] <= max_num_player
        assert x.shape[1] <= input_sequence_size

        pad0 = max_num_player - x.shape[0]
        pad1 = input_sequence_size - x.shape[1]
        x = np.pad(x,
                   ((0, pad0), (pad1, 0), (0, 0)),
                   mode='constant',
                   constant_values=val)
        return x

    def target_pad(self, x, max_num_player=MAX_PLAYER_NUM, output_sequence_size=48, val=0.0):

        assert x.shape[0] <= max_num_player
        assert x.shape[1] <= output_sequence_size

        pad0 = max_num_player - x.shape[0]
        pad1 = output_sequence_size - x.shape[1]
        x = np.pad(x,
                   ((0, pad0), (0, pad1), (0, 0)),
                   mode='constant',
                   constant_values=val)
        return x

    def __getitem__(self, i):

        i_df = self.test_input.copy()
        o_df = self.test.copy()

        i_info_df = i_df[["nfl_id", "player_role", "player_to_predict"]].drop_duplicates().reset_index(drop=True)
        i_nfl_ids = i_df["nfl_id"].drop_duplicates().values.copy()
        o_nfl_ids = o_df["nfl_id"].drop_duplicates().values.copy()

        # Generate indexes to reorder nfl_ids
        # The reordering rules are as follows:
        # - IDs that match those in o_nfl_ids take the same order as in o_nfl_ids
        # - IDs that do not match are appended afterward while preserving their original order
        # This allows us to simply take the first len(o_nfl_ids) elements
        sort_order = [i for i, x in enumerate(i_nfl_ids) if x in set(o_nfl_ids)]
        sort_order += [i for i, x in enumerate(i_nfl_ids) if x not in set(o_nfl_ids)]

        # Get sizes
        num_player_input = len(i_nfl_ids)
        num_frame_input = len(i_df) // num_player_input
        num_frame_output = o_df["frame_id"].max()
        num_player_output = len(o_nfl_ids)

        i_data = i_df[src_feat_names].values.astype(np.float32)
        ## o_data = o_df[["x", "y"]].values.astype(np.float32)
        i_data = i_data.reshape(num_player_input, num_frame_input, -1)
        ## o_data = o_data.reshape(num_player_output, num_frame_output, -1)

        receiver_xy = i_df[i_df["player_role"] == "Targeted Receiver"][["x", "y"]].values.astype(np.float32)
        passer_xy = i_df[i_df["player_role"] == "Passer"][["x", "y"]].values.astype(np.float32)
        ball_land_xy = i_df.loc[0, ["ball_land_x", "ball_land_y"]].values.astype(np.float32)

        # Sort players in the order: target players first, then others
        i_data = i_data[sort_order]
        i_nfl_ids = [i_nfl_ids[i] for i in sort_order]
        i_info_df = i_info_df.iloc[sort_order].reset_index(drop=True)

        data = {
            "i_data": i_data,
            ## "o_data": o_data,
            "o_df": o_df,
            "i_info_df": i_info_df,
            "receiver_xy": receiver_xy,
            "passer_xy": passer_xy,
            "ball_land_xy": ball_land_xy,
        }


        i_info_df = data["i_info_df"].copy()
        ball_land_xy = data["ball_land_xy"].copy()
        receiver_idx = i_info_df.index[i_info_df["player_role"] == "Targeted Receiver"].tolist()[0]
        passer_idx = i_info_df.index[i_info_df["player_role"] == "Passer"].tolist()[0]

        # Data augumentaiton start
        frame_shift_size = 0.0

        # get size
        num_player_input, num_frame_input, _ = i_data.shape
        ## num_player_output, num_frame_output, _ = o_data.shape

        #
        receiver_xy = i_data[receiver_idx][:, [src_feat_idx["x"], src_feat_idx["y"]]]
        passer_xy = i_data[passer_idx][ :, [src_feat_idx["x"], src_feat_idx["y"]]]

        # extruct features
        ext_feat_idx = self.ext_feat_idx
        ext_feat_names = self.ext_feat_names
        ext_data = np.zeros((num_player_input, num_frame_input, len(ext_feat_names)), dtype=np.float32)
        if "x" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["x"]] = i_data[:, :, src_feat_idx["x"]].copy()
        if "y" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["y"]] = i_data[:, :, src_feat_idx["y"]].copy()
        if "ori_sin" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["ori_sin"]] = np.sin(np.deg2rad(i_data[:, :, src_feat_idx["o"]]))
        if "ori_cos" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["ori_cos"]] = np.cos(np.deg2rad(i_data[:, :, src_feat_idx["o"]]))
        if "dir_sin_s" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["dir_sin_s"]] = np.sin(np.deg2rad(i_data[:, :, src_feat_idx["dir"]])) * i_data[:, :, src_feat_idx["s"]]
        if "dir_cos_s" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["dir_cos_s"]] = np.cos(np.deg2rad(i_data[:, :, src_feat_idx["dir"]])) * i_data[:, :, src_feat_idx["s"]]
        if "ball_land_x_dx" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["ball_land_x_dx"]] = ball_land_xy[0] - i_data[:, :, src_feat_idx["x"]]
        if "ball_land_y_dy" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["ball_land_y_dy"]] = ball_land_xy[1] - i_data[:, :, src_feat_idx["y"]]
        if "receiver_dx" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["receiver_dx"]] = receiver_xy[:,0] - i_data[:, :, src_feat_idx["x"]]
        if "receiver_dy" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["receiver_dy"]] = receiver_xy[:, 1] - i_data[:, :, src_feat_idx["y"]]

        # Vector from ball landing point to the passer
        # Angle of that vector relative to the x-axis
        # Rotate coordinates so the passer direction aligns with the positive x-axis
        #rot_passer_xy  = ball_land_xy - tmp_xy

        rot_passer_xy = passer_xy[-1] - ball_land_xy # パサーの最終座標とボール座標の差
        theta = np.arctan2(rot_passer_xy[1], rot_passer_xy[0])
        tmp_xy = i_data[:, :][:,:,[src_feat_idx["x"],src_feat_idx["y"]]].copy()
        shift_xy = tmp_xy - ball_land_xy
        rot_x = np.cos(theta) * shift_xy[:,:,0] + np.sin(theta) * shift_xy[:,:,1]
        rot_y = -np.sin(theta) * shift_xy[:,:, 0] + np.cos(theta) * shift_xy[:,:, 1]
        if "rot_x" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["rot_x"]] = rot_x
        if "rot_y" in self.ext_feat_names:
            ext_data[:, :, ext_feat_idx["rot_y"]] = rot_y
        
        # The target is defined as the offset from the final input coordinates
        # target_xy = o_data - i_data[:num_player_output, -1:, [src_feat_idx["x"], src_feat_idx["y"]]]
        last_xy = i_data[:num_player_output, -1:, :]
        last_xy = last_xy[:, :, [src_feat_idx["x"], src_feat_idx["y"]]]
        ## target_xy = o_data - last_xy

        # Extract the last 20 frames
        ext_data = ext_data[:, -self.input_sequence_size:].copy()

        # Target data for auxiliary loss､speed and acceleration
        ## last_xy_input = i_data[:len(o_data), -2:][:,:,[src_feat_idx["x"], src_feat_idx["y"]]].copy()
        ## targets_s = np.diff(np.concatenate((last_xy_input, o_data), axis=1), axis=1)
        ## targets_a = np.diff(targets_s, axis=1)
        ## targets_s = targets_s[:, 1:].copy()
        ## target_sa = np.concatenate((targets_s, targets_a), axis=2)


        # nomarlize
        ext_data = self.feature_scaling(ext_data)
        ## target_xy = self.target_scaling(target_xy)
        ## target_sa = (target_sa - self.target_sa_mean) / self.target_sa_std

        # fill nan to 0
        ext_data = np.nan_to_num(ext_data)
        ## target_xy = np.nan_to_num(target_xy)
        ## target_sa = np.nan_to_num(target_sa)

        # make constant shapes
        ext_data = self.feat_pad(ext_data)
        ## target_xy = self.target_pad(target_xy)
        ## target_sa = self.target_pad(target_sa)

        # make static features start
        player_role = [player_role_to_idx[role] for role in i_info_df["player_role"]]
        player_role_onehot = np.identity(len(player_role_to_idx))[player_role]

        num_frame_output_feat = (num_frame_output - self.num_frame_output_mean) / self.num_frame_output_std

        player_last_xy = i_data[:, -1][:,[src_feat_idx["x"], src_feat_idx["y"]]].copy()
        player_last_xy = (player_last_xy - self.player_last_xy_mean) / self.player_last_xy_std

        passer_last_xy = passer_xy[-1]
        passer_last_xy = (passer_last_xy - self.passer_last_xy_mean) / self.passer_last_xy_std
        ball_land_xy = (ball_land_xy - self.ball_land_xy_mean) / self.ball_land_xy_std

        static_feat = np.zeros((MAX_PLAYER_NUM, 12), dtype=np.float32)
        static_feat[:len(player_role_onehot), :4] = player_role_onehot
        static_feat[:, 4] = num_frame_output_feat # # 全プレイヤーに同じ num_frame_output_feat をブロードキャストする設計
        static_feat[:, 5] = (frame_shift_size - 3.0) / 3.0
        static_feat[:, 6:8] = passer_last_xy # 全プレイヤーに同じ passer_xy をブロードキャストする設計
        static_feat[:, 8:10] = ball_land_xy
        static_feat[:len(player_last_xy), 10:12] = player_last_xy
        # make static features end

        # to Channel, Player, Length
        dynamic_feat = np.transpose(ext_data, (2, 0, 1))
        ## target_xy = np.transpose(target_xy, (2, 0, 1))  # 2pl
        ## target_sa = np.transpose(target_sa, (2, 0, 1))  # 2pl

        # make mask for loss func
        ## target_mask = np.zeros_like(target_xy)
        ## target_mask[:, :num_player_output, :num_frame_output] = 1.0

        # add batch dimension
        dynamic_feat = dynamic_feat[None]
        static_feat = static_feat[None]
        num_player_input = torch.tensor(num_player_input).reshape(1).long()
        num_player_output = torch.tensor(num_player_output).reshape(1).long()
        num_frame_output = torch.tensor(num_frame_output).reshape(1).long()
        last_xy = np.transpose(last_xy,(2,0,1))
        last_xy = torch.tensor(last_xy)

        dataset = {"dynamic_feat": torch.tensor(dynamic_feat).float(),
                   "static_feat": torch.tensor(static_feat).float(),
                   ## "target_xy": torch.tensor(target_xy).float(),
                   ##  "target_sa": torch.tensor(target_sa).float(),
                   ## "target_mask": torch.tensor(target_mask).float(),
                   "num_frame_output": num_frame_output,
                   "num_player_output": num_player_output,
                   "num_player_input": num_player_input,
                   "last_xy" : last_xy
                   }

        return dataset

## Load emsenble models 
+ All ensemble members use the same model architecture.
+ Diversity is achieved by varying the input dynamic features.



In [7]:
def load_ems_models(model_paths , feature_names_ ):

    ems_models_ = {}
    ems_models_["models"] = []
    ems_models_["feature_names"] = feature_names_.copy()
    for model_path in model_paths:
        model = Model(
                input_dim=len(feature_names_),
                output_sequence_size=CFG.output_sequence_size,
                loss_fn=torch.nn.Identity(),
                model_param=CFG.model_param,
            )
        print(model_path)
        model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=True)
        model = model.eval()
        model = model.to(CFG.device)
        model = torch.compile(model)
        ems_models_["models"].append(model)
    print(f"{len(ems_models_['models'])=}")
    print(f"{ems_models_['feature_names']=}")
    return ems_models_




In [8]:
# Best feature set(base)
# Private: 0.46471 wo tta, 5fold seed emsenble
model_num = 15 # 
model_paths = []
model_paths += sorted(Path('/kaggle/input/nfl2026-1st-place/models_base').glob('*.pt'))
model_paths = model_paths[:model_num]
ems_models_base = load_ems_models(model_paths,
                                   ["x", "y",                             # coordiante
                                    "ori_sin", "ori_cos",                 # orientation
                                     "ball_land_x_dx", "ball_land_y_dy",  # distance to ball
                                     "receiver_dx", "receiver_dy",        # distance to receiver
                                     "dir_sin_s", "dir_cos_s",            # speed vector
                                     #"rot_x", "rot_y"                     # pass-aligned coordinate
                                   ] )

/kaggle/input/nfl2026-1st-place/models_base/0_0_best.pt
/kaggle/input/nfl2026-1st-place/models_base/0_1_best.pt
/kaggle/input/nfl2026-1st-place/models_base/0_2_best.pt
/kaggle/input/nfl2026-1st-place/models_base/0_3_best.pt
/kaggle/input/nfl2026-1st-place/models_base/0_4_best.pt
/kaggle/input/nfl2026-1st-place/models_base/1_0_best.pt
/kaggle/input/nfl2026-1st-place/models_base/1_1_best.pt
/kaggle/input/nfl2026-1st-place/models_base/1_2_best.pt
/kaggle/input/nfl2026-1st-place/models_base/1_3_best.pt
/kaggle/input/nfl2026-1st-place/models_base/1_4_best.pt
/kaggle/input/nfl2026-1st-place/models_base/2_0_best.pt
/kaggle/input/nfl2026-1st-place/models_base/2_1_best.pt
/kaggle/input/nfl2026-1st-place/models_base/2_2_best.pt
/kaggle/input/nfl2026-1st-place/models_base/2_3_best.pt
/kaggle/input/nfl2026-1st-place/models_base/2_4_best.pt
len(ems_models_['models'])=15
ems_models_['feature_names']=['x', 'y', 'ori_sin', 'ori_cos', 'ball_land_x_dx', 'ball_land_y_dy', 'receiver_dx', 'receiver_dy', 'd

In [9]:
# Base features excluding ball and receiver coordinates
# Private: 0.46511 wo tta, 5fold seed emsenble

model_paths = []
model_paths += sorted(Path('/kaggle/input/nfl2026-1st-place/models_norb').glob('*.pt'))
model_paths = model_paths[:model_num]
ems_models_norb = load_ems_models(model_paths,
                                   ["x", "y",                             # coordiante
                                    "ori_sin", "ori_cos",                 # orientation
                                     #"ball_land_x_dx", "ball_land_y_dy",  # distance to ball
                                     #"receiver_dx", "receiver_dy",        # distance to receiver
                                     "dir_sin_s", "dir_cos_s",            # speed vector
                                     #"rot_x", "rot_y"                     # pass-aligned coordinate
                                   ] )

/kaggle/input/nfl2026-1st-place/models_norb/0_0_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/0_1_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/0_2_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/0_3_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/0_4_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/1_0_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/1_1_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/1_2_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/1_3_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/1_4_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/2_0_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/2_1_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/2_2_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/2_3_best.pt
/kaggle/input/nfl2026-1st-place/models_norb/2_4_best.pt
len(ems_models_['models'])=15
ems_models_['feature_names']=['x', 'y', 'ori_sin', 'ori_cos', 'dir_sin_s', 'dir_cos_s']


In [10]:
# Base features with pass-aligned coordinates added
# Private: 0.46946 wo tta, 5fold seed emsenble

model_paths = []
model_paths += sorted(Path('/kaggle/input/nfl2026-1st-place/models_rot').glob('*.pt'))
model_paths = model_paths[:model_num]
ems_models_rot = load_ems_models(model_paths,
                                   ["x", "y",                             # coordiante
                                    "ori_sin", "ori_cos",                 # orientation
                                     "ball_land_x_dx", "ball_land_y_dy",  # distance to ball
                                     "receiver_dx", "receiver_dy",        # distance to receiver
                                     "dir_sin_s", "dir_cos_s",            # speed vector
                                     "rot_x", "rot_y"                     # pass-aligned coordinate
                                   ] )

/kaggle/input/nfl2026-1st-place/models_rot/0_0_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/0_1_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/0_2_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/0_3_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/0_4_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/1_0_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/1_1_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/1_2_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/1_3_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/1_4_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/2_0_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/2_1_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/2_2_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/2_3_best.pt
/kaggle/input/nfl2026-1st-place/models_rot/2_4_best.pt
len(ems_models_['models'])=15
ems_models_['feature_names']=['x', 'y', 'ori_sin', 'ori_cos', 'ball_land_x_dx', 'ball_land_y_dy', 'receiver_dx', 'receiver_dy', 'dir_sin_s', 'dir

In [11]:

def __predict(test, test_input, ems_models, use_flipud=False):

    test = test.copy()
    test_input = test_input.copy()
    
    # data, last_xy_input = make_data(test, test_input, ems_models["feature_names"], use_flipud)
    test_input[["dir", "o"]] = (90.0 - test_input[["dir", "o"]]) % 360.0
    test_input = unify_play_direction_input(test_input)

    # flipud for test time aug
    if use_flipud:
        test_input = apply_flipud(test_input)
        
    CFG.ext_feat_names = ems_models["feature_names"]
    T0 = TestDataset(test_input,test,CFG)
    data = T0[0]


    # return test
    data = {k: v.to(CFG.device) for k, v in data.items()}
    last_xy = data["last_xy"].cpu().numpy().copy()
    length = data["num_frame_output"].item()
    num_player_output = data["num_player_output"].item()

    preds = []
    for model in ems_models["models"]:
        with torch.inference_mode():
            # pred, _ = model(data)
            pred = model.inference(data)
        
        pred = pred.detach().cpu().numpy()
        pred = pred[0, :, :num_player_output]
        # print(f"{pred.shape=}")

        # Adjust number of frames
        pred_length = pred.shape[2]
        if pred_length == length:
            pass
        elif pred_length < length:
            # If there are not enough, padding with the last value
            last = pred[:, :, -1:]
            pad = np.repeat(last, length - pred_length, axis=2)
            pred = np.concatenate([pred, pad], axis=2)
        else:
            # If it overflows, cut it off
            pred = pred[:, :, :length]
        #print(f"{pred.shape=}")

        # rescaling
        target_xy_mean = T0.target_mean.copy()
        target_xy_std = T0.target_std.copy()
        target_xy_mean = target_xy_mean.reshape(-1,1,1)
        target_xy_std = target_xy_std.reshape(-1,1,1)
        pred_xy = (pred * target_xy_std) + target_xy_mean
        pred_xy = pred_xy + last_xy

    
        # If it is upside down, return it to its original position.
        if use_flipud:
            pred_xy[1] = FIELD_HEIGHT - pred_xy[1]
        
        # If it is flipped 180 degrees, return it to its original position.
        if test_input.iloc[0]["play_direction"] == "left":
            pred_xy[0] = FIELD_WIDTH - pred_xy[0]
            pred_xy[1] = FIELD_HEIGHT - pred_xy[1]

        # clip within the field
        pred_xy[0] = np.clip(pred_xy[0], 0.0 - field_mergine, FIELD_WIDTH + field_mergine)
        pred_xy[1] = np.clip(pred_xy[1], 0.0 - field_mergine, FIELD_HEIGHT + field_mergine)

        preds.append(pred_xy)


    return preds


def unify_play_direction_input(df):
    df = df.copy()
    play_direction_mask = df["play_direction"] == "left"
    df.loc[play_direction_mask, "x"] = FIELD_WIDTH - df.loc[play_direction_mask, "x"]
    df.loc[play_direction_mask, "y"] = FIELD_HEIGHT - df.loc[play_direction_mask, "y"]
    df.loc[play_direction_mask, "ball_land_x"] = FIELD_WIDTH - df.loc[play_direction_mask, "ball_land_x"]
    df.loc[play_direction_mask, "ball_land_y"] = FIELD_HEIGHT - df.loc[play_direction_mask, "ball_land_y"]
    df.loc[play_direction_mask, "dir"] = (df.loc[play_direction_mask, "dir"] + 180.0) % 360.0
    df.loc[play_direction_mask, "o"] = (df.loc[play_direction_mask, "o"] + 180.0) % 360.0
    return df


def apply_flipud(df):
    df["y"] = FIELD_HEIGHT - df["y"]  # y
    df["ball_land_y"] = FIELD_HEIGHT - df["ball_land_y"]  # y
    df["dir"] = (360.0 - df["dir"]) % 360.0  #
    df["o"] = (360.0 - df["o"]) % 360.0  #
    return df

## Predict 

In [12]:
def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    """Replace this function with your inference code.
    You can return either a Pandas or Polars dataframe, though Polars is recommended for performance.
    Each batch of predictions (except the very first) must be returned within 5 minutes of the batch features being provided.
    """

    test = test.to_pandas()
    test_input = test_input.to_pandas()

    # Ensemble of each model and TTA (flip_ud)
    preds_list = []
    preds_list += __predict(test, test_input, ems_models_base, use_flipud=False)
    preds_list += __predict(test, test_input, ems_models_base, use_flipud=True) 
    preds_list += __predict(test, test_input, ems_models_norb, use_flipud=False)
    preds_list += __predict(test, test_input, ems_models_norb, use_flipud=True) 
    preds_list += __predict(test, test_input, ems_models_rot, use_flipud=False) 
    preds_list += __predict(test, test_input, ems_models_rot, use_flipud=True) 

    # average    
    preds = np.nanmean(preds_list, axis=0)
    
    pred_x = preds[0].reshape(-1)
    pred_y = preds[1].reshape(-1)
    pred_df = test.copy()
    pred_df["x"] = pred_x
    pred_df["y"] = pred_y        
    predictions = pred_df[["x","y"]]

    # print(predictions.head(10))
        
    # assert isinstance(predictions, (pd.DataFrame, pl.DataFrame))
    # assert len(predictions) == len(test)
    return predictions


# When your notebook is run on the hidden test set, inference_server.serve must be called within 10 minutes of the notebook starting
# or the gateway will throw an error. If you need more than 15 minutes to load your model you can do so during the very
# first `predict` call, which does not have the usual 5 minute response deadline.
inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/nfl-big-data-bowl-2026-prediction/',))

```

           x          y
0  88.372645  34.296334
1  88.682691  34.302290
2  89.029010  34.362235
3  89.394600  34.473462
4  89.761709  34.626082
5  90.129382  34.828265
6  90.478011  35.068240
7  90.798560  35.338241
8  91.084786  35.672578
9  91.326305  36.014470

-----
x          y
0   88.340002  34.295362
1   88.658124  34.306949
2   89.008861  34.359296
3   89.377896  34.461250
4   89.753430  34.610330
5   90.125167  34.804468
6   90.476736  35.044811
7   90.804052  35.320734
8   91.091416  35.634183
9   91.338377  35.982372
10  91.554852  36.349165
11  85.702511  38.414149
12  85.772344  39.010405
```